# Motivation Figure: OpenVLA-OFT mismatch ratio on perturbed instructions

**Goal.** Quantify how often a publicly-available OpenVLA-OFT checkpoint produces actions that are *inconsistent with the perturbed task instruction* on our controlled perturbation suite.

**Mismatch ratio definition.** For each perturbed sample, compare the action vector predicted from the *original* instruction with the action vector predicted from the *perturbed* instruction:

$$\mathrm{mismatch} = \mathbb{1}\bigl[ \| a_{perturbed} - a_{original} \|_2 < \tau \bigr]$$

If the model treats the perturbation as a no-op (predictions barely change), the perturbation has not influenced behaviour — that is the failure mode we report.

**Prerequisites.**
1. `scripts/03_extract_object_poses.py` has been run on `libero_spatial`.
2. `scripts/03b_generate_perturbation_suite.py` has produced `data/perturbation_suite/`.
3. The OpenVLA-OFT checkpoint (`openvla/openvla-7b-oft`) is downloaded — see `third_party/openvla-oft/README.md`.
4. CUDA-enabled GPU available (≥ 24 GB VRAM recommended).

If any prerequisite is missing, the notebook will print a clear `[SKIP]` line and stop early.

## 1. Environment / data check

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

REPO_ROOT = Path('..').resolve()
PERTURB_DIR = REPO_ROOT / 'data' / 'perturbation_suite'
MANIFEST = PERTURB_DIR / 'manifest.json'

print(f'[ENV] python={sys.version.split()[0]}')
print(f'[REPO] {REPO_ROOT}')
print(f'[DATA] perturb_dir={PERTURB_DIR}')

if not MANIFEST.exists():
    print('[SKIP] perturbation suite manifest not found.')
    print('       Run: python scripts/03b_generate_perturbation_suite.py ...')
    raise SystemExit

In [ ]:
import torch

if not torch.cuda.is_available():
    print('[SKIP] CUDA not available — OpenVLA-OFT inference requires a GPU.')
    raise SystemExit

print(f'[CUDA] device={torch.cuda.get_device_name(0)}')
print(f'[CUDA] memory={torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 2. Load the OpenVLA-OFT checkpoint

We use the publicly available checkpoint as published in the OpenVLA-OFT paper. No fine-tuning here — this measures *out-of-the-box* sensitivity to instruction perturbations.

In [ ]:
try:
    from transformers import AutoModelForVision2Seq, AutoProcessor
except ImportError:
    print('[SKIP] transformers not installed.')
    raise SystemExit

CHECKPOINT_ID = 'openvla/openvla-7b-oft'      # public checkpoint

print(f'[CKPT] loading {CHECKPOINT_ID} ...')
processor = AutoProcessor.from_pretrained(CHECKPOINT_ID, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    CHECKPOINT_ID,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
).to('cuda').eval()
print(f'[CKPT] loaded.')

## 3. Mismatch-ratio computation

For each perturbed sample in the manifest, predict actions for both `original_instruction` and `perturbed_instruction`, then compute whether the L2 difference is below threshold `tau`.

In [ ]:
TAU = 0.02              # paper §V.B mismatch threshold
MAX_SAMPLES = 200       # bound for notebook runtime

manifest = json.loads(MANIFEST.read_text())
samples_dir = PERTURB_DIR / 'samples'

all_records = []
for task_id, by_type in manifest.items():
    for ptype, files in by_type.items():
        if not isinstance(files, list):
            continue
        for fname in files:
            all_records.append((task_id, ptype, samples_dir / fname))

print(f'[DATA] {len(all_records)} candidate samples; capping at {MAX_SAMPLES}')
all_records = all_records[:MAX_SAMPLES]

In [ ]:
@torch.no_grad()
def predict_action(image: np.ndarray, instruction: str) -> np.ndarray:
    """Run the OpenVLA-OFT model on a single (image, instruction) pair."""
    prompt = f'In: What action should the robot take to {instruction}?\nOut:'
    inputs = processor(prompt, image, return_tensors='pt').to('cuda', dtype=torch.bfloat16)
    action = model.predict_action(**inputs, unnorm_key='libero_spatial', do_sample=False)
    return np.asarray(action).reshape(-1).astype(np.float32)


def load_image_for_sample(record_path: Path) -> np.ndarray:
    """Load the image associated with a perturbation record.
    
    For visual_style records, the perturbed image is on disk; for text-only
    records, fall back to a placeholder neutral background.
    """
    record = json.loads(record_path.read_text())
    if 'perturbed_image_path' in record:
        return np.load(record_path.parent / record['perturbed_image_path'])
    return np.full((128, 128, 3), 128, dtype=np.uint8)

In [ ]:
from collections import defaultdict

results = defaultdict(list)

for i, (task_id, ptype, record_path) in enumerate(all_records):
    record = json.loads(record_path.read_text())
    original = record['original_instruction']
    perturbed = record.get('perturbed_instruction', original)

    image = load_image_for_sample(record_path)

    a_orig = predict_action(image, original)
    a_pert = predict_action(image, perturbed)

    diff = float(np.linalg.norm(a_orig - a_pert))
    is_mismatch = diff < TAU      # treats perturbation as no-op = mismatch
    results[ptype].append({'task_id': task_id, 'diff': diff, 'mismatch': is_mismatch})

    if (i + 1) % 25 == 0:
        print(f'  processed {i + 1} / {len(all_records)}')

## 4. Per-perturbation-type mismatch ratio + figure

In [ ]:
import matplotlib.pyplot as plt

ratios = {}
for ptype, recs in results.items():
    if not recs:
        continue
    ratios[ptype] = sum(r['mismatch'] for r in recs) / len(recs)

for ptype, ratio in ratios.items():
    print(f'  {ptype:<28s} mismatch_ratio = {ratio:.3f}  (n={len(results[ptype])})')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(ratios.keys()), list(ratios.values()))
ax.set_ylabel('Mismatch ratio')
ax.set_xlabel('Perturbation type')
ax.set_title('OpenVLA-OFT robustness to controlled instruction perturbations')
ax.set_ylim(0, 1)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('motivation_figure.png', dpi=150)
plt.show()

## 5. Persist raw measurements

Write the per-sample results to JSON for inclusion in the paper's supplementary material.

In [ ]:
out = {
    'tau': TAU,
    'checkpoint': CHECKPOINT_ID,
    'mismatch_ratios': ratios,
    'per_sample': {ptype: recs for ptype, recs in results.items()},
}
with open('motivation_results.json', 'w') as f:
    json.dump(out, f, indent=2)

print('Wrote motivation_results.json')